In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from fuzzywuzzy import process
import warnings 

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
warnings.filterwarnings ('ignore')
pd.set_option ('display.width', None)
pd.set_option ('display.max_rows', 100)
pd.set_option ('display.max_columns', 50)

# Create `Football_Player_Profile` Dataset

## Read *base_info_--.csv*

In [14]:
leagues = ['EPL','Laliga','Bundesliga','SerieA','Ligue1']
info = []
for league in leagues:
    path = f"ds108/{league}/base_info_{league}.csv"
    base_info_ = pd.read_csv (path)
    base_info_.insert (2, 'League',f'{league}')
    print (f"{league}: {base_info_.shape}")
    info.append (base_info_)
base_info = pd.concat (info, ignore_index=True)
# df = df.rename (columns={'ten_cu':'ten_moi'})
# Đổi cột Name --> Player để đồng nhất
base_info = base_info.rename (columns={'Name':'Player'})
base_info.drop (columns=['Unnamed: 0'], inplace=True)
base_info

EPL: (564, 7)
Laliga: (587, 8)
Bundesliga: (487, 8)
SerieA: (625, 8)
Ligue1: (543, 8)


,Player,Club,League,Age,Height,National,Positions
0,Mohamed Salah,Liverpool,EPL,32 years old (15-06-1992),175cm,Egypt,"Attacking Midfielder (Centre, Left, Right), Fo..."
1,Bukayo Saka,Arsenal,EPL,23 years old (05-09-2001),178cm,England,"Defender (Left), Midfielder (Centre, Left, Right)"
2,Alexander Isak,Newcastle,EPL,25 years old (21-09-1999),192cm,Sweden,"Attacking Midfielder (Left), Forward"
3,Matheus Cunha,Wolves,EPL,25 years old (27-05-1999),183cm,Brazil,"Attacking Midfielder (Centre, Left), Forward"
4,Erling Haaland,Manchester City,EPL,24 years old (21-07-2000),194cm,Norway,Forward
...,...,...,...,...,...,...,...
2801,Bernard Nguene,Nice,Ligue1,18 years old (04-08-2006),178cm,Cameroon,Forward
2802,Paul Akouokou,Lyon,Ligue1,27 years old (20-12-1997),181cm,Ivory Coast,Defensive Midfielder (C)
2803,Bamo Meïté,Montpellier,Ligue1,23 years old (03-12-2001),183cm,NaN,Defender (Centre)
2804,Élysée Logbo,Le Havre,Ligue1,20 years old (06-05-2004),188cm,France,Forward


In [24]:
n = base_info["Player"].unique ()
n.shape

(2655,)

## Read *data_summer_EPL.csv*

In [25]:
info = []
for league in leagues:
    path = f"ds108/{league}/data_summary_{league}.csv"
    data_sum = pd.read_csv (path)
    info.append (data_sum)
    
data_summary = pd.concat (info, ignore_index=True)

data_summary.drop (columns=['Unnamed: 0'], inplace=True)
SpG = data_summary ["SpG"]
data_summary.drop ('SpG', axis=1, inplace=True)

SpG = pd.DataFrame (SpG)
print ("SpG shape:",SpG.shape)
data_summary

SpG shape: (2806, 1)


,Player,Apps,Mins,Goals,Assists,Yel,Red,PS%,AerialsWon,MotM,Rating
0,Mohamed Salah,32,2840,27,18,1,-,74.3,0.3,10,7.75
1,Bukayo Saka,16(3),1372,6,10,3,-,84.3,0.4,6,7.59
2,Alexander Isak,29,2351,21,6,1,-,75.2,0.9,5,7.44
3,Matheus Cunha,24(3),2168,14,4,3,-,77.9,0.4,8,7.37
4,Erling Haaland,28,2484,21,3,2,-,66.4,1.8,6,7.37
...,...,...,...,...,...,...,...,...,...,...,...
2801,Bernard Nguene,0(1),13,-,-,-,-,100,-,-,5.87
2802,Paul Akouokou,0(2),19,-,-,1,-,66.7,-,-,5.79
2803,Bamo Meïté,6,413,-,-,2,1,88.6,1,-,5.76
2804,Élysée Logbo,0(1),26,-,-,-,-,75,-,-,5.67


In [26]:
n = data_summary["Player"].unique ()
n.shape

(2655,)

## *Football_Player_Profile*

In [27]:
# Football_Player_Profile = pd.merge (base_info, data_summary, on='Player')
Football_Player_Profile = pd.concat ([base_info, data_summary.drop (columns=['Player'])],axis=1)
Football_Player_Profile = Football_Player_Profile.rename (columns={'Yel':'YelC','Red':'RedC','AerialsWon':'ArlW'})

Football_Player_Profile

,Player,Club,League,Age,Height,National,Positions,Apps,Mins,Goals,Assists,YelC,RedC,PS%,ArlW,MotM,Rating
0,Mohamed Salah,Liverpool,EPL,32 years old (15-06-1992),175cm,Egypt,"Attacking Midfielder (Centre, Left, Right), Fo...",32,2840,27,18,1,-,74.3,0.3,10,7.75
1,Bukayo Saka,Arsenal,EPL,23 years old (05-09-2001),178cm,England,"Defender (Left), Midfielder (Centre, Left, Right)",16(3),1372,6,10,3,-,84.3,0.4,6,7.59
2,Alexander Isak,Newcastle,EPL,25 years old (21-09-1999),192cm,Sweden,"Attacking Midfielder (Left), Forward",29,2351,21,6,1,-,75.2,0.9,5,7.44
3,Matheus Cunha,Wolves,EPL,25 years old (27-05-1999),183cm,Brazil,"Attacking Midfielder (Centre, Left), Forward",24(3),2168,14,4,3,-,77.9,0.4,8,7.37
4,Erling Haaland,Manchester City,EPL,24 years old (21-07-2000),194cm,Norway,Forward,28,2484,21,3,2,-,66.4,1.8,6,7.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2801,Bernard Nguene,Nice,Ligue1,18 years old (04-08-2006),178cm,Cameroon,Forward,0(1),13,-,-,-,-,100,-,-,5.87
2802,Paul Akouokou,Lyon,Ligue1,27 years old (20-12-1997),181cm,Ivory Coast,Defensive Midfielder (C),0(2),19,-,-,1,-,66.7,-,-,5.79
2803,Bamo Meïté,Montpellier,Ligue1,23 years old (03-12-2001),183cm,NaN,Defender (Centre),6,413,-,-,2,1,88.6,1,-,5.76
2804,Élysée Logbo,Le Havre,Ligue1,20 years old (06-05-2004),188cm,France,Forward,0(1),26,-,-,-,-,75,-,-,5.67


In [28]:
Football_Player_Profile.shape 

(2806, 17)

# Create `Offensive_Profile` Dataset

## Read *data_details_Goals_EPL.csv* 
> Goals

In [29]:
# Các cách đổi thứ tự cột
'''
Cách 1: df = df[['C','A','B']]
Cách 2: Sử dụng reindex () theo axis=1
    df = df.reindex (columns=['C','A','B'])
Cách 3: col = df.pop ('C')
        df.insert (0, 'C', col) --> Xoá rồi chèn
'''

"\nCách 1: df = df[['C','A','B']]\nCách 2: Sử dụng reindex () theo axis=1\n    df = df.reindex (columns=['C','A','B'])\nCách 3: col = df.pop ('C')\n        df.insert (0, 'C', col) --> Xoá rồi chèn\n"

In [30]:
info = []
for league in leagues:
    Goal = pd.read_csv (f'ds108/{league}/data_details_Goals_{league}.csv')
    Goal = Goal.reindex (columns =[
                                'Total','OutOfBox',
                                'SixYardBox','PenaltyArea'
                            ])
    Goal = Goal.rename (columns = {
                        'Total':'TotGs',
                        'OutOfBox':'OOGBs',
                        'SixYardBox':'SYBGs',
                        'PenaltyArea':'PAGs'
                    })
    info.append (Goal)
    
Goals = pd.concat (info).reset_index (drop=True)
SpG = SpG.reset_index (drop=True)
# Gộp thuộc tính SpG 
Goals = pd.concat ([SpG,Goals], axis=1)

print ("Goals Dataset's Shape:",Goals.shape)
Goals

Goals Dataset's Shape: (2806, 5)


,SpG,TotGs,OOGBs,SYBGs,PAGs
0,3.4,0.8,-,0.2,0.7
1,2.7,0.3,-,0.1,0.3
2,2.9,0.7,0.1,0.3,0.3
3,3.3,0.5,0.2,-,0.3
4,3.6,0.8,0.1,0.1,0.5
...,...,...,...,...,...
2801,-,-,-,-,-
2802,-,-,-,-,-
2803,0.2,-,-,-,-
2804,1,-,-,-,-


## Read *data_details_Shots_--.csv*
> Shots

In [31]:
info = []
for league in leagues:
    Shot = pd.read_csv (f'ds108/{league}/data_details_Shots_{league}.csv') 
    Shot = Shot.rename (columns = {
        'Total' : 'TotSh',
        'OutOfBox' : 'OOBSh',
        'SixYardBox' : 'SYBSh',
        'PenaltyArea' : 'PASh'
    })
    info.append (Shot)
Shots = pd.concat (info, ignore_index=True)
Shots

,TotSh,OOBSh,SYBSh,PASh
0,3.4,0.3,0.4,2.7
1,2.7,0.6,0.2,1.9
2,2.9,0.5,0.3,2
3,3.3,1.6,0.1,1.6
4,3.6,0.3,0.7,2.6
...,...,...,...,...
2801,-,-,-,-
2802,-,-,-,-
2803,0.2,0.2,-,-
2804,1,-,-,1


## Read *data_details_Dribbles_--.csv*
> Dribbles

In [32]:
info = []
for league in leagues:
    Dribble = pd.read_csv (f'ds108/{league}/data_details_Dribbles_{league}.csv') 
    Dribble = Dribble [['Total Dribbles','Unsuccessful','Successful']]
    Dribble.rename (columns={
        'Total Dribbles' : 'TotDrib',
        'Unsuccessful' : 'UnDrib',
        'Successful' : 'SucDrib'
    }, inplace=True)
    info.append (Dribble)
Dribbles = pd.concat (info).reset_index (drop=True)
Dribbles

,TotDrib,UnDrib,SucDrib
0,3.7,2,1.6
1,4.3,2.4,1.9
2,2.9,1.4,1.4
3,4.4,2.5,1.9
4,1.1,0.8,0.4
...,...,...,...
2801,1,1,-
2802,-,-,-
2803,0.2,-,0.2
2804,-,-,-


## Read *data_details_Possession loss_--.csv*
> Possession_Loss

In [33]:
info = []
for league in leagues:
    possloss = pd.read_csv (f"ds108/{league}/data_details_Possession loss_{league}.csv")
    possloss.rename (columns = {
            'UnsuccessfulTouches' : "UnTch",
            'Dispossessed' : 'Dispo'
        }, inplace=True)
    info.append (possloss)
    
Possession_Loss = pd.concat (info).reset_index (drop=True)
Possession_Loss

,UnTch,Dispo
0,3.1,1.8
1,1.4,1.9
2,2,1.7
3,2.5,1.6
4,1.4,0.9
...,...,...
2801,1,-
2802,0.5,-
2803,0.5,0.5
2804,-,3


## Read *data_details_Aerial_--.csv*
> Aerial

In [34]:
info = []
for league in leagues:
    Aeri = pd.read_csv (f'ds108/{league}/data_details_Aerial_{league}.csv') 
    Aeri.rename (columns={
        'Total' : 'TotAD',
        'Won' : 'WonAD',
        'Lost' : 'LostAD'
    }, inplace=True)
    print (f"{league}: {Aeri.shape}")
    info.append (Aeri)
Aerial = pd.concat (info, ignore_index=True)
Aerial

EPL: (564, 3)
Laliga: (587, 3)
Bundesliga: (487, 3)
SerieA: (625, 3)
Ligue1: (543, 3)


,TotAD,WonAD,LostAD
0,0.7,0.3,0.4
1,1.4,0.4,0.9
2,2.5,0.9,1.6
3,1.2,0.4,0.9
4,3.4,1.8,1.6
...,...,...,...
2801,2,-,2
2802,-,-,-
2803,2.2,1,1.2
2804,2,-,2


## Create *Offensive_Profile* DataSet

In [35]:
Offensive_Profile = pd.concat ([Goals, Shots, Dribbles, Possession_Loss, Aerial], axis=1)
Offensive_Profile

,SpG,TotGs,OOGBs,SYBGs,PAGs,TotSh,OOBSh,SYBSh,PASh,TotDrib,UnDrib,SucDrib,UnTch,Dispo,TotAD,WonAD,LostAD
0,3.4,0.8,-,0.2,0.7,3.4,0.3,0.4,2.7,3.7,2,1.6,3.1,1.8,0.7,0.3,0.4
1,2.7,0.3,-,0.1,0.3,2.7,0.6,0.2,1.9,4.3,2.4,1.9,1.4,1.9,1.4,0.4,0.9
2,2.9,0.7,0.1,0.3,0.3,2.9,0.5,0.3,2,2.9,1.4,1.4,2,1.7,2.5,0.9,1.6
3,3.3,0.5,0.2,-,0.3,3.3,1.6,0.1,1.6,4.4,2.5,1.9,2.5,1.6,1.2,0.4,0.9
4,3.6,0.8,0.1,0.1,0.5,3.6,0.3,0.7,2.6,1.1,0.8,0.4,1.4,0.9,3.4,1.8,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2801,-,-,-,-,-,-,-,-,-,1,1,-,1,-,2,-,2
2802,-,-,-,-,-,-,-,-,-,-,-,-,0.5,-,-,-,-
2803,0.2,-,-,-,-,0.2,0.2,-,-,0.2,-,0.2,0.5,0.5,2.2,1,1.2
2804,1,-,-,-,-,1,-,-,1,-,-,-,-,3,2,-,2


# Create `Defensive_Profile` Dataset

In [36]:
info = []
for league in leagues:
    tackle = pd.read_csv (f'ds108/{league}/data_details_Tackles_{league}.csv')
    interception = pd.read_csv (f'ds108/{league}/data_details_Interception_{league}.csv')
    foul = pd.read_csv (f'ds108/{league}/data_details_Fouls_{league}.csv')
    offside = pd.read_csv (f'ds108/{league}/data_details_Offsides_{league}.csv')
    clearance = pd.read_csv (f'ds108/{league}/data_details_Clearances_{league}.csv')
    blocked = pd.read_csv (f'ds108/{league}/data_details_Blocks_{league}.csv')
    save = pd.read_csv (f'ds108/{league}/data_details_Saves_{league}.csv')
    save = save.reindex (columns=[
        'Total','SixYardBox',
        'PenaltyArea','OutOfBox'
    ])
    offensive = pd.concat ([tackle, interception, foul, offside, clearance,
                           blocked, save],axis=1) 
    offensive.columns = [
                            'TotTkl', 'DribPast','AttTkl','Intercpt',
                            'Fouled','Fouls','COF','Clr','BlkSh','BlkCr',
                            'BlkPs','TotSav', 'OOBSav', 'SYBSav','PASav'
                        ]
    print (f"{league}: {offensive.shape}")
    info.append (offensive)
offensive
Defensive_Profile = pd.concat (info, ignore_index=True)
Defensive_Profile
    

EPL: (564, 15)
Laliga: (587, 15)
Bundesliga: (487, 15)
SerieA: (625, 15)
Ligue1: (543, 15)


,TotTkl,DribPast,AttTkl,Intercpt,Fouled,Fouls,COF,Clr,BlkSh,BlkCr,BlkPs,TotSav,OOBSav,SYBSav,PASav
0,0.6,0.2,0.8,0.2,1,0.7,0.5,0.2,-,-,0.3,-,-,-,-
1,1.2,0.5,1.7,0.1,1.8,0.7,0.2,0.3,-,0.1,0.9,-,-,-,-
2,0.4,0.2,0.7,0.1,0.4,0.8,0.7,0.5,0.1,-,0.5,-,-,-,-
3,1.2,0.9,2.1,0.6,2.2,1.4,0.3,0.7,-,0.1,0.6,-,-,-,-
4,0.3,0.1,0.5,0.2,0.4,0.8,0.1,0.8,-,-,0.4,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2801,-,-,-,-,-,1,-,-,-,-,-,-,-,-,-
2802,0.5,-,0.5,0.5,-,1.5,-,-,-,-,-,-,-,-,-
2803,0.7,0.2,0.8,0.5,0.7,1.2,-,2.2,0.5,-,-,-,-,-,-
2804,-,-,-,-,-,-,-,-,-,-,-,-,-,-,-


# Create `Passing_Profile` Dataset

In [37]:
info = []
for league in leagues:
    passes = pd.read_csv (f'ds108/{league}/data_details_Passes_{league}.csv')
    key_pass = pd.read_csv (f'ds108/{league}/data_details_Key passes_{league}.csv')
    assists = pd.read_csv (f'ds108/{league}/data_details_Assists_{league}.csv')
    assists = assists.reindex (columns=[
        'Total','Cross', 'Corner','Throughball',
        'Freekick','Throwin','Other'
    ])
    df = pd.concat ([passes, key_pass, assists],axis=1) 
    df.columns = [
                    'TotPs', 'AccLB','InAccLB','AccSP',
                    'InAccSP','TotKPs','LKPs','SKPs',
                    'TotAss', 'CrAss','CorAss','ThrbAss',
                    'FreAss','ThrInAss','OthAss'
                ]
    print (f"{league}: {df.shape}")
    info.append (df)
Passing_Profile = pd.concat (info, ignore_index=True)
Passing_Profile

EPL: (564, 15)
Laliga: (587, 15)
Bundesliga: (487, 15)
SerieA: (625, 15)
Ligue1: (543, 15)


,TotPs,AccLB,InAccLB,AccSP,InAccSP,TotKPs,LKPs,SKPs,TotAss,CrAss,CorAss,ThrbAss,FreAss,ThrInAss,OthAss
0,31,0.7,0.7,22.3,7.3,2.3,0.2,2.2,0.6,0.1,-,-,-,-,0.5
1,24.1,0.3,0.5,20.1,3.3,2.3,0.6,1.7,0.5,0.3,0.2,0.1,-,-,0.2
2,19.2,0.4,0.4,14,4.4,1.3,0.1,1.2,0.2,-,-,-,-,-,0.2
3,28.8,1.4,1.2,21,5.1,1.7,0.2,1.5,0.1,-,-,-,-,-,0.1
4,12,0.1,-,7.9,4,0.9,-,0.9,0.1,-,-,-,-,-,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2801,2,-,-,2,-,-,-,-,-,-,-,-,-,-,-
2802,7.5,0.5,-,4.5,2.5,-,-,-,-,-,-,-,-,-,-
2803,26.3,0.5,0.5,22.8,2.5,-,-,-,-,-,-,-,-,-,-
2804,4,-,-,3,1,-,-,-,-,-,-,-,-,-,-


# Create `Football_Player_Full` Dataset

In [38]:
Football_Player_Full = pd.concat ([Football_Player_Profile,Offensive_Profile,Defensive_Profile,Passing_Profile],axis=1)
# Football_Player_Full = Football_Player_Full.sort_values (by=['League','Club']).reset_index(drop=True)
Football_Player_Full

,Player,Club,League,Age,Height,National,Positions,Apps,Mins,Goals,Assists,YelC,RedC,PS%,ArlW,MotM,Rating,SpG,TotGs,OOGBs,SYBGs,PAGs,TotSh,OOBSh,SYBSh,...,Fouls,COF,Clr,BlkSh,BlkCr,BlkPs,TotSav,OOBSav,SYBSav,PASav,TotPs,AccLB,InAccLB,AccSP,InAccSP,TotKPs,LKPs,SKPs,TotAss,CrAss,CorAss,ThrbAss,FreAss,ThrInAss,OthAss
0,Mohamed Salah,Liverpool,EPL,32 years old (15-06-1992),175cm,Egypt,"Attacking Midfielder (Centre, Left, Right), Fo...",32,2840,27,18,1,-,74.3,0.3,10,7.75,3.4,0.8,-,0.2,0.7,3.4,0.3,0.4,...,0.7,0.5,0.2,-,-,0.3,-,-,-,-,31,0.7,0.7,22.3,7.3,2.3,0.2,2.2,0.6,0.1,-,-,-,-,0.5
1,Bukayo Saka,Arsenal,EPL,23 years old (05-09-2001),178cm,England,"Defender (Left), Midfielder (Centre, Left, Right)",16(3),1372,6,10,3,-,84.3,0.4,6,7.59,2.7,0.3,-,0.1,0.3,2.7,0.6,0.2,...,0.7,0.2,0.3,-,0.1,0.9,-,-,-,-,24.1,0.3,0.5,20.1,3.3,2.3,0.6,1.7,0.5,0.3,0.2,0.1,-,-,0.2
2,Alexander Isak,Newcastle,EPL,25 years old (21-09-1999),192cm,Sweden,"Attacking Midfielder (Left), Forward",29,2351,21,6,1,-,75.2,0.9,5,7.44,2.9,0.7,0.1,0.3,0.3,2.9,0.5,0.3,...,0.8,0.7,0.5,0.1,-,0.5,-,-,-,-,19.2,0.4,0.4,14,4.4,1.3,0.1,1.2,0.2,-,-,-,-,-,0.2
3,Matheus Cunha,Wolves,EPL,25 years old (27-05-1999),183cm,Brazil,"Attacking Midfielder (Centre, Left), Forward",24(3),2168,14,4,3,-,77.9,0.4,8,7.37,3.3,0.5,0.2,-,0.3,3.3,1.6,0.1,...,1.4,0.3,0.7,-,0.1,0.6,-,-,-,-,28.8,1.4,1.2,21,5.1,1.7,0.2,1.5,0.1,-,-,-,-,-,0.1
4,Erling Haaland,Manchester City,EPL,24 years old (21-07-2000),194cm,Norway,Forward,28,2484,21,3,2,-,66.4,1.8,6,7.37,3.6,0.8,0.1,0.1,0.5,3.6,0.3,0.7,...,0.8,0.1,0.8,-,-,0.4,-,-,-,-,12,0.1,-,7.9,4,0.9,-,0.9,0.1,-,-,-,-,-,0.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2801,Bernard Nguene,Nice,Ligue1,18 years old (04-08-2006),178cm,Cameroon,Forward,0(1),13,-,-,-,-,100,-,-,5.87,-,-,-,-,-,-,-,-,...,1,-,-,-,-,-,-,-,-,-,2,-,-,2,-,-,-,-,-,-,-,-,-,-,-
2802,Paul Akouokou,Lyon,Ligue1,27 years old (20-12-1997),181cm,Ivory Coast,Defensive Midfielder (C),0(2),19,-,-,1,-,66.7,-,-,5.79,-,-,-,-,-,-,-,-,...,1.5,-,-,-,-,-,-,-,-,-,7.5,0.5,-,4.5,2.5,-,-,-,-,-,-,-,-,-,-
2803,Bamo Meïté,Montpellier,Ligue1,23 years old (03-12-2001),183cm,NaN,Defender (Centre),6,413,-,-,2,1,88.6,1,-,5.76,0.2,-,-,-,-,0.2,0.2,-,...,1.2,-,2.2,0.5,-,-,-,-,-,-,26.3,0.5,0.5,22.8,2.5,-,-,-,-,-,-,-,-,-,-
2804,Élysée Logbo,Le Havre,Ligue1,20 years old (06-05-2004),188cm,France,Forward,0(1),26,-,-,-,-,75,-,-,5.67,1,-,-,-,-,1,-,-,...,-,-,-,-,-,-,-,-,-,-,4,-,-,3,1,-,-,-,-,-,-,-,-,-,-


# Create CSV Files From Datasets

*Warnings: Don't run!*

In [39]:
Football_Player_Profile.to_csv ('Football_Player_Profile.csv')
Offensive_Profile.to_csv ('Offensive_Profile.csv')
Defensive_Profile.to_csv ('Defensive_Profensive.csv')
Passing_Profile.to_csv ('Passing_Profile.csv')
Football_Player_Full.to_csv ('Football_Player_Full_Dataset.csv')

In [ ]:
epl_clubs_2425 = [
    'AFC Bournemouth', 'Arsenal FC', 'Aston Villa', 'Brentford FC',
    'Brighton & Hove Albion', 'Chelsea FC', 'Crystal Palace',
    'Everton FC', 'Fulham FC', 'Ipswich Town', 'Leicester City',
    'Liverpool FC', 'Manchester City', 'Manchester United',
    'Newcastle United', 'Nottingham Forest', 'Southampton FC',
    'Tottenham Hotspur', 'West Ham United', 'Wolverhampton Wanderers'
]
df_not_in_epl = EPL[~EPL['Club'].isin(epl_clubs_2425)].copy()
EPL = EPL[EPL['Club'].isin(epl_clubs_2425)].copy()
df_not_in_epl

In [ ]:
# Xoá trắng, chuyển thường
def clean_text(text):
    if isinstance(text, str):
        return text.strip().lower()
    return ""

import re
def normalize_name(name):
    if isinstance(name, str):
        name = name.lower().strip()
        name = re.sub(r'\b(fc|club|cf|sc|afc)\b', '', name)  # bỏ hậu tố
        name = re.sub(r'[^a-zA-Z\s]', '', name)  # bỏ ký tự đặc biệt
        name = re.sub(r'\s+', ' ', name)  # bỏ khoảng trắng thừa
        return name.strip()
    return ""

EPL['club_clean'] = EPL['Club'].apply(normalize_name)
df_all['club_clean'] = df_all['CLB'].apply(normalize_name)

EPL['player_clean'] = EPL['Player'].apply(clean_text)
df_all['player_clean'] = df_all['Player'].apply(clean_text)

merged_df = pd.merge(EPL, df_all, on='player_clean', how='left', suffixes=('_df_all', '_EPL'))
merged_df